In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
from IPython.display import Image
try:
  filename = take_photo()
  print('Saved to {}'.format(filename))

  # Show the image which was just taken.
  display(Image(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

<IPython.core.display.Javascript object>

NotReadableError: Device in use


In [ ]:
from IPython.display import Image
try:
  filename = take_photo()
  print('Saved to {}'.format(filename))

  # Show the image which was just taken.
  display(Image(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

<IPython.core.display.Javascript object>

NotReadableError: Device in use


After installing these libraries, you can paste your existing code into a new cell and execute it. Please note that the real-time camera detection part (`cv2.VideoCapture(0)`) might behave differently or require specific setup in a Jupyter environment compared to a local Python script or Colab.

In [ ]:
from IPython.display import display, Javascript, Image
import google.colab
from base64 import b64decode, b64encode
import numpy as np
import cv2
import io

def process_frame_for_gender(frame_bytes_b64):
    """Decodes, processes with gender detection, and re-encodes a single frame."""
    global model, face_cascade # Access global variables from the previous training cell

    # Decode base64 to numpy array
    nparr = np.frombuffer(b64decode(frame_bytes_b64), np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    if frame is None:
        return "" # Return empty string if frame decoding fails

    # Perform gender detection (logic copied from original cell Ahd0ceXaiV4Q, section 4)
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

    for (x, y, w, h) in faces:
        face_roi = frame[y:y+h, x:x+w]
        try:
            face_resized = cv2.resize(face_roi, (64, 64))
            face_flat = face_resized.flatten().reshape(1, -1)
            probabilities = model.predict_proba(face_flat)

            confidence_male = probabilities[0][0]
            confidence_female = probabilities[0][1]

            if confidence_male > confidence_female:
                label = "Male"
                color = (255, 0, 0) # Blue
                val = confidence_male
            else:
                label = "Female"
                color = (0, 0, 255) # Pinkish
                val = confidence_female

            label_text = f"{label}: {val*100:.1f}%"

            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.rectangle(frame, (x, y - 30), (x + w, y), color, -1)
            cv2.putText(frame, label_text, (x + 5, y - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        except Exception as e:
            pass # Ignore errors in face processing, e.g., if face_roi is too small

    # Encode processed frame back to base64 JPEG
    _, encimg = cv2.imencode('.jpeg', frame)
    processed_frame_b64 = b64encode(encimg.tobytes()).decode('utf-8')
    return processed_frame_b64

def start_camera_stream_colab():
    """Starts a JavaScript camera stream, sends frames to Python for processing, and displays them."""
    js_code = Javascript('''
        var video = document.createElement('video');
        video.style.display = 'block';
        var stream;
        var canvas = document.createElement('canvas');
        var ctx = canvas.getContext('2d');
        var img = document.createElement('img'); // For displaying processed frames
        document.body.appendChild(img); // Append img element to display processed frames

        async function setupAndStartStream() {
            try {
                stream = await navigator.mediaDevices.getUserMedia({video: true});
                video.srcObject = stream;
                await video.play();

                google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

                async function captureAndProcessFrame() {
                    if (!stream.active) { // Stop if stream is no longer active
                        console.log("Stream stopped.");
                        return;
                    }

                    canvas.width = video.videoWidth;
                    canvas.height = video.videoHeight;
                    ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
                    var imageData = canvas.toDataURL('image/jpeg', 0.8); // Get base64 image data

                    // Send to Python for processing
                    var processedImageData = await google.colab.kernel.invokeFunction(
                        'process_image_from_js_callback', [imageData.split(',')[1]], {} // Function name must match Python defined one
                    );

                    // Display the processed image from Python
                    if (processedImageData.data && processedImageData.data[0]) {
                        img.src = 'data:image/jpeg;base64,' + processedImageData.data[0];
                    }

                    // Request next frame to continue the loop
                    requestAnimationFrame(captureAndProcessFrame);
                }
                requestAnimationFrame(captureAndProcessFrame);
            } catch (err) {
                console.error("Error accessing camera: ", err);
                alert("Could not start camera. Please ensure camera permissions are granted and no other application is using it.");
            }
        }

        async function stopStream() {
            if (stream) {
                stream.getTracks().forEach(track => track.stop());
                video.remove();
                canvas.remove();
                img.remove();
                console.log("Camera stream stopped.");
            }
        }

        // Expose functions to Python for calling
        google.colab.kernel.defineFunction('start_camera_js', setupAndStartStream);
        google.colab.kernel.defineFunction('stop_camera_js', stopStream);
    ''')
    display(js_code)

    # Register the Python function that JavaScript will call
    google.colab.kernel.defineFunction('process_image_from_js_callback', process_frame_for_gender)

    # Start the stream from Python via JavaScript
    google.colab.output.eval_js('start_camera_js()')
    print("Camera stream started. Look for the video output below.")
    print("To stop the stream, run: `google.colab.output.eval_js('stop_camera_js()')` in a new cell.")

In [ ]:
start_camera_stream_colab()

<IPython.core.display.Javascript object>

AttributeError: module 'google.colab' has no attribute 'kernel'

In [ ]:
# Run this cell to stop the camera stream.
google.colab.output.eval_js('stop_camera_js()')

MessageError: ReferenceError: stop_camera_js is not defined

In [ ]:
import cv2
import os
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- 1. Download & Setup Dataset ---
print("⬇ Downloading Gender dataset from Kaggle... (This happens once)")

# Using the requested dataset: 'gmlmrinalini/genderdetectionface'
dataset_path = kagglehub.dataset_download("gmlmrinalini/genderdetectionface")

print(f"✅ Dataset downloaded to: {dataset_path}")

# --- 2. Data Loading ---
print("📂 Loading images and training model... (Grab a coffee, this takes ~2 mins)")

data = []
labels = []

# This dataset typically contains folders 'man' and 'woman'
# We map them: 0 = Male, 1 = Female
categories = {
    "man": 0,
    "woman": 1
}

# The dataset structure often has subfolders like 'dataset1/train'
# We walk through to find where the images actually are
base_dir = dataset_path

# Helper to find the actual data folder if nested
found_data = False
for root, dirs, files in os.walk(dataset_path):
    if "man" in dirs and "woman" in dirs:
        base_dir = root
        found_data = True
        break

if not found_data:
    # Fallback/Assumption if walk fails to find exact structure
    print("⚠  Note: Could not auto-locate 'man'/'woman' folders. using root.")

for folder_name, label in categories.items():
    folder_path = os.path.join(base_dir, folder_name)

    if not os.path.exists(folder_path):
        print(f"⚠ Warning: Could not find folder '{folder_name}' in {base_dir}")
        continue

    print(f"   Processing {folder_name}...")

    # Limit to 1000 images per class to speed up training on your i5 laptop
    images = os.listdir(folder_path)[:1000]

    for file in images:
        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path)

        if img is not None:
            # Resize is crucial for SVM (must be fixed size)
            img = cv2.resize(img, (64, 64))
            data.append(img.flatten())     # Flatten: 64x64x3 -> 12288 features
            labels.append(label)

X = np.array(data)
y = np.array(labels)
print(X.shape, y.shape)

if len(X) == 0:
    print("❌ Error: No images were loaded. Check the download path structure.")
    exit()

print(f"✅ Loaded {len(X)} images. Starting Training...")

# --- 3. Training the Model ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# probability=True is required for the confidence percentage later
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Calculate accuracy
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"🎉 Model trained! Accuracy: {acc*100:.2f}%")

# --- 4. Real-Time Detection ---
print("🎥 Starting Camera... Press 'q' to quit.")

# Load Haar Cascade (Face Detector)
haar_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(haar_path)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Face Detection needs Grayscale
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

    # 2. Loop through every face found
    for (x, y, w, h) in faces:
        # Get the face ROI (Region of Interest)
        face_roi = frame[y:y+h, x:x+w]

        # 3. Preprocess exactly like the Training Data (Resize -> Flatten)
        try:
            face_resized = cv2.resize(face_roi, (64, 64))
            face_flat = face_resized.flatten().reshape(1, -1)

            # 4. Predict
            probabilities = model.predict_proba(face_flat)
            # probabilities returns [[prob_male, prob_female]]

            confidence_male = probabilities[0][0]
            confidence_female = probabilities[0][1]

            if confidence_male > confidence_female:
                label = "Male"
                color = (255, 0, 0) # Blue
                val = confidence_male
            else:
                label = "Female"
                color = (0, 0, 255) # Pinkish
                val = confidence_female

            label_text = f"{label}: {val*100:.1f}%"

            # 5. Draw UI
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            # Background bar for text
            cv2.rectangle(frame, (x, y - 30), (x + w, y), color, -1)
            cv2.putText(frame, label_text, (x + 5, y - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        except Exception as e:
            pass

    cv2.imshow("Gender Detector (RandomForestClassifier)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

⬇ Downloading Gender dataset from Kaggle... (This happens once)


100%|██████████| 136M/136M [00:02<00:00, 66.8MB/s]

Extracting files...


✅ Dataset downloaded to: /root/.cache/kagglehub/datasets/gmlmrinalini/genderdetectionface/versions/1
📂 Loading images and training model... (Grab a coffee, this takes ~2 mins)
   Processing man...
   Processing woman...
(340, 12288) (340,)
✅ Loaded 340 images. Starting Training...
🎉 Model trained! Accuracy: 86.76%
🎥 Starting Camera... Press 'q' to quit.
